# MATE: experimentos con Isolation

Este notebook reúne evidencia reproducible para el informe y la defensa de la Parte 2. Usa directamente las implementaciones de `board.py`, `mate_agents.py`, `mate_evaluations.py`, `random_agent.py` y `experiments_mate.py`; los algoritmos no se vuelven a implementar aquí.

## 1. El problema de Isolation

Isolation es un juego determinista, secuencial, completamente observable y de suma cero. En cada turno, el jugador activo se mueve exactamente una casilla en una de las ocho direcciones y luego elimina una casilla libre. Una casilla eliminada no puede volver a utilizarse. Pierde el jugador que comienza su turno sin movimientos válidos.

Una acción es el par `(dirección, casilla_a_eliminar)`. Por tanto, el factor de ramificación combina los movimientos disponibles con todas las eliminaciones legales.

In [1]:
import sys
print(sys.executable)

/Users/gabo/Desktop/Facultad/Inteligencia Artificial/Practicos/Practico 6 - IA/.venv/bin/python


In [2]:
from pathlib import Path
import random
import sys
import time

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# Permite abrir el notebook desde Isolation/ o desde la raíz del repositorio.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "board.py").exists():
    candidate = PROJECT_DIR / "Isolation"
    if not (candidate / "board.py").exists():
        raise FileNotFoundError("No se encontró el directorio Isolation")
    PROJECT_DIR = candidate
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from board import Board
from mate_agents import AlphaBetaAgent, ExpectimaxAgent, MinimaxAgent
from mate_evaluations import EVALUATION_FUNCTIONS, balanced_eval
from random_agent import RandomAgent
import experiments_mate

RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)
SEED = 2026
print(f"Proyecto: {PROJECT_DIR}")
print(f"Funciones disponibles: {', '.join(EVALUATION_FUNCTIONS)}")

ModuleNotFoundError: No module named 'pandas'

## 2. Tablero 4×4

Fijamos la semilla para obtener siempre el mismo estado inicial. `B` representa al jugador 1, `R` al jugador 2 y `·` una casilla libre.

In [ ]:
random.seed(SEED)
board = Board((4, 4))
symbols = {0: "·", 1: "B", 2: "R", 3: "X"}
board_view = pd.DataFrame(board.grid).replace(symbols)
board_view.index.name = "fila"
board_view.columns.name = "columna"
display(board_view)

## 3. Acciones posibles del jugador 1

Se consulta la API existente de `Board`. La casilla eliminable ya se calcula después de realizar el movimiento.

In [ ]:
direction_names = {
    0: "arriba", 1: "abajo", 2: "izquierda", 3: "derecha",
    4: "arriba-izquierda", 5: "arriba-derecha",
    6: "abajo-izquierda", 7: "abajo-derecha",
}
possible_actions = board.get_possible_actions(1)
actions_df = pd.DataFrame(
    [
        {
            "acción": action,
            "dirección": direction_names[action[0]],
            "casilla eliminada": action[1],
        }
        for action in possible_actions
    ]
)
print(f"Jugador 1 tiene {len(possible_actions)} acciones legales.")
display(actions_df)

## 4. Validación de acciones legales

Los cuatro agentes reciben el mismo tablero. Una acción es válida si pertenece a la lista generada por `Board.get_possible_actions(1)`.

In [ ]:
agents = {
    "Random": RandomAgent(player=1, seed=SEED),
    "Minimax": MinimaxAgent(player=1, depth=1, evaluation_function=balanced_eval, seed=SEED),
    "AlphaBeta": AlphaBetaAgent(player=1, depth=1, evaluation_function=balanced_eval, seed=SEED),
    "Expectimax": ExpectimaxAgent(player=1, depth=1, evaluation_function=balanced_eval, seed=SEED),
}
validation_rows = []
for name, agent in agents.items():
    action = agent.next_action(board)
    validation_rows.append({"agente": name, "acción": action, "es legal": action in possible_actions})

validation_df = pd.DataFrame(validation_rows)
display(validation_df)
assert validation_df["es legal"].all(), "Algún agente devolvió una acción ilegal"
print("✓ Todos los agentes devolvieron acciones legales.")

## 5. Minimax frente a Alpha-Beta (profundidad 2)

Ambos usan `balanced_eval` y el mismo orden de acciones. Deben producir la misma decisión Minimax; Alpha-Beta puede evitar evaluar ramas que no cambiarán el resultado. `expanded_nodes` cuenta todos los nodos visitados, incluidas hojas.

In [ ]:
comparison_agents = {
    "Minimax": MinimaxAgent(1, depth=2, evaluation_function=balanced_eval, seed=SEED),
    "AlphaBeta": AlphaBetaAgent(1, depth=2, evaluation_function=balanced_eval, seed=SEED),
}
comparison_rows = []
for name, agent in comparison_agents.items():
    action = agent.next_action(board)
    metrics = agent.last_metrics
    comparison_rows.append({
        "agente": name,
        "acción elegida": action,
        "nodos expandidos": metrics.expanded_nodes,
        "ramas podadas": metrics.pruned_branches,
        "tiempo (s)": metrics.execution_time,
    })
comparison_df = pd.DataFrame(comparison_rows)
display(comparison_df)
assert comparison_df.loc[0, "acción elegida"] == comparison_df.loc[1, "acción elegida"]
assert comparison_df.loc[1, "nodos expandidos"] <= comparison_df.loc[0, "nodos expandidos"]
print("✓ Minimax y Alpha-Beta coinciden; Alpha-Beta no visita más nodos.")

## 6. Ejemplo de Expectimax

Expectimax maximiza en los turnos propios y promedia uniformemente los valores de todas las acciones legales del rival. No utiliza poda Alpha-Beta.

In [ ]:
expectimax = ExpectimaxAgent(1, depth=2, evaluation_function=balanced_eval, seed=SEED)
expectimax_action = expectimax.next_action(board)
expectimax_example = pd.DataFrame([{
    "acción elegida": expectimax_action,
    "es legal": expectimax_action in possible_actions,
    "profundidad": expectimax.last_metrics.search_depth,
    "nodos expandidos": expectimax.last_metrics.expanded_nodes,
    "tiempo (s)": expectimax.last_metrics.execution_time,
}])
display(expectimax_example)

## 7. Comparación de funciones de evaluación

Cada heurística se evalúa desde las perspectivas de ambos jugadores sobre el mismo estado no terminal. Los valores están limitados entre −1 y 1 para que una estimación no supere a una victoria terminal demostrada.

In [ ]:
evaluation_rows = []
for name, evaluation_function in EVALUATION_FUNCTIONS.items():
    evaluator = AlphaBetaAgent(1, depth=1, evaluation_function=evaluation_function, seed=SEED)
    action = evaluator.next_action(board)
    evaluation_rows.append({
        "función": name,
        "valor para jugador 1": evaluation_function(board, 1),
        "valor para jugador 2": evaluation_function(board, 2),
        "acción AlphaBeta (d=1)": action,
        "nodos expandidos": evaluator.last_metrics.expanded_nodes,
    })
evaluation_df = pd.DataFrame(evaluation_rows)
display(evaluation_df)

## 8. Resultados de múltiples partidas

El CSV se genera con `experiments_mate.py`. Por ejemplo: `poetry run python experiments_mate.py --suite core --games 20 --depths 1 2 3`.

In [ ]:
csv_path = RESULTS_DIR / "mate_experiments.csv"
if not csv_path.exists():
    raise FileNotFoundError(
        f"No existe {csv_path}. Ejecuta experiments_mate.py antes de continuar."
    )
results = pd.read_csv(csv_path)
print(f"Archivo: {csv_path}")
print(f"Filas: {len(results)} | Columnas: {len(results.columns)}")
display(results.head())

## 9. Tabla resumen

Agrupamos por agente, función de evaluación y profundidad. La tasa de victoria se recalcula como victorias totales divididas entre partidas totales.

In [ ]:
summary = (
    results.groupby(["agent_name", "evaluation_function", "search_depth"], as_index=False)
    .agg(
        configuraciones=("matchup", "nunique"),
        partidas=("games", "sum"),
        victorias=("wins", "sum"),
        derrotas=("losses", "sum"),
        longitud_media=("average_game_length", "mean"),
        tiempo_medio_s=("average_decision_time_seconds", "mean"),
        nodos_medios=("average_expanded_nodes", "mean"),
        podas_medias=("average_pruned_branches", "mean"),
    )
)
summary["tasa_victoria"] = summary["victorias"] / summary["partidas"]
summary = summary[[
    "agent_name", "evaluation_function", "search_depth",
    "configuraciones", "partidas", "victorias", "derrotas",
    "tasa_victoria", "longitud_media", "tiempo_medio_s",
    "nodos_medios", "podas_medias",
]]
display(summary.round(4))

## 10. Gráficos

Los siguientes gráficos se guardan como PNG dentro de `results/`. Se utilizan agregaciones simples para que sean fáciles de explicar en el informe.

In [ ]:
win_rate_plot = results.groupby("agent_name", as_index=False)[["wins", "games"]].sum()
win_rate_plot["win_rate"] = win_rate_plot["wins"] / win_rate_plot["games"]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(win_rate_plot["agent_name"], win_rate_plot["win_rate"], color="#4C78A8")
ax.set(title="Tasa de victoria por agente", xlabel="Agente", ylabel="Tasa de victoria", ylim=(0, 1))
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
win_rate_path = RESULTS_DIR / "mate_win_rate.png"
fig.savefig(win_rate_path, dpi=150)
plt.show()

In [ ]:
search_results = results[results["search_depth"] > 0]
decision_time_plot = (
    search_results.groupby(["agent_name", "search_depth"])["average_decision_time_seconds"]
    .mean().unstack("agent_name")
)
ax = decision_time_plot.plot(kind="bar", figsize=(8, 4.5))
ax.set(title="Tiempo medio de decisión por profundidad", xlabel="Profundidad", ylabel="Segundos")
ax.grid(axis="y", alpha=0.25)
ax.figure.tight_layout()
decision_time_path = RESULTS_DIR / "mate_decision_time.png"
ax.figure.savefig(decision_time_path, dpi=150)
plt.show()

In [ ]:
expanded_nodes_plot = (
    search_results.groupby(["agent_name", "search_depth"])["average_expanded_nodes"]
    .mean().unstack("agent_name")
)
ax = expanded_nodes_plot.plot(kind="bar", figsize=(8, 4.5))
ax.set(title="Nodos expandidos por profundidad", xlabel="Profundidad", ylabel="Nodos medios por decisión")
ax.grid(axis="y", alpha=0.25)
ax.figure.tight_layout()
expanded_nodes_path = RESULTS_DIR / "mate_expanded_nodes.png"
ax.figure.savefig(expanded_nodes_path, dpi=150)
plt.show()

In [ ]:
pruned_plot = (
    results[results["agent_name"] == "AlphaBeta"]
    .groupby("search_depth")["average_pruned_branches"].mean()
)
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.bar(pruned_plot.index.astype(str), pruned_plot.values, color="#F58518")
ax.set(title="Ramas podadas por Alpha-Beta", xlabel="Profundidad", ylabel="Ramas medias por decisión")
ax.grid(axis="y", alpha=0.25)
fig.tight_layout()
pruned_branches_path = RESULTS_DIR / "mate_pruned_branches.png"
fig.savefig(pruned_branches_path, dpi=150)
plt.show()

saved_plots = [win_rate_path, decision_time_path, expanded_nodes_path, pruned_branches_path]
print("Gráficos guardados:")
for plot_path in saved_plots:
    print(f"- {plot_path}")

## 11. Plantilla de conclusión para el informe

> En los experimentos realizados sobre un tablero 4×4, **[agente]** obtuvo la mayor tasa de victoria (**[valor]%**) con profundidad **[d]** y la función **[heurística]**. Al aumentar la profundidad, el tiempo de decisión **[aumentó/disminuyó]** debido al crecimiento del árbol. Minimax y Alpha-Beta mantuvieron decisiones equivalentes, pero Alpha-Beta visitó **[número]** nodos frente a **[número]** de Minimax y podó **[número]** ramas, confirmando su ventaja computacional. Expectimax resultó **[más/menos]** adecuado contra Random porque modela al rival mediante una distribución uniforme. Entre las evaluaciones, **[heurística]** ofreció el mejor equilibrio entre movilidad propia y bloqueo del rival. Estos resultados están limitados por **[cantidad de partidas, profundidad y tamaño del tablero]**, por lo que conviene interpretar pequeñas diferencias con cautela.

Antes de entregar, sustituir los campos entre corchetes por los valores observados en la tabla y los gráficos.